# เทรน YOLO11 บน Colab (Smart Fridge · CV)

**ก่อนเริ่ม (ครั้งแรกครั้งเดียว)** อัปโหลดไฟล์ 2 ไฟล์ขึ้น Google Drive ไว้ที่ `MyDrive/smart-fridge-cv/`
- `foodseg_yolo_split.zip` (1.91 GB, มี train/val/test แล้ว)
- `train.py` (จาก `scripts/train.py` ในเครื่อง)

**Runtime → Change runtime type → L4 GPU** (กิน ~1.94 CU/ชม.)

ผลเทรนเซฟลง Drive (`MyDrive/smart-fridge-cv/runs/detect/<name>`) ทุก epoch ถ้า session หลุดให้รันเซลล์ 1–4 ใหม่ แล้วใช้เซลล์ **Resume**

⚠️ เซลล์สุดท้ายตัด runtime ทิ้งเพื่อหยุดกิน CU อย่าเปิดค้างไว้

In [ ]:
# 1) ตรวจ GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) mount Drive และตรวจว่ามีไฟล์ครบ
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/smart-fridge-cv')
for f in ['foodseg_yolo_split.zip', 'train.py']:
    p = DRIVE / f
    print(('OK ' if p.exists() else 'MISSING ') + str(p))

In [ ]:
# 3) คัดลอก dataset มาไว้ดิสก์ของ Colab แล้วแตก (อ่านจาก /content เร็วกว่าอ่านจาก Drive มาก)
!cp "{DRIVE}/foodseg_yolo_split.zip" /content/
!unzip -q -o /content/foodseg_yolo_split.zip -d /content/
!for s in train val test; do echo $s $(ls /content/foodseg_yolo/images/$s | wc -l); done
# ต้องได้ train 10055 / val 1959 / test 1317

In [ ]:
# 4) ติดตั้ง ultralytics (torch มีใน Colab อยู่แล้ว)
!pip install -q ultralytics
import torch, ultralytics
print(torch.__version__, torch.cuda.get_device_name(0), 'ultralytics', ultralytics.__version__)

In [ ]:
# 5) เทรน (แก้ NAME / MODEL ตามการทดลองใน experiments.md)
NAME = 'baseline_n'
MODEL = 'yolo11n.pt'
PROJECT = DRIVE / 'runs' / 'detect'

!python "{DRIVE}/train.py" --name {NAME} --model {MODEL} \
    --data /content/foodseg_yolo/data.yaml --project "{PROJECT}" \
    --epochs 50 --patience 15 --batch 32 --workers 8 --cache ram

In [ ]:
# Resume: ใช้เมื่อ session หลุดกลางคัน (รันเซลล์ 1–4 ก่อน แล้วค่อยรันเซลล์นี้ ไม่ต้องรันเซลล์ 5)
NAME = 'baseline_n'
LAST = DRIVE / 'runs' / 'detect' / NAME / 'weights' / 'last.pt'
print(LAST, LAST.exists())
!python "{DRIVE}/train.py" --name {NAME} --resume "{LAST}"

In [ ]:
# 6) ดูผลคร่าวๆ (ไฟล์ทั้งหมดอยู่ใน Drive แล้ว)
import pandas as pd
from IPython.display import Image, display
run = DRIVE / 'runs' / 'detect' / NAME
df = pd.read_csv(run / 'results.csv')
df.columns = df.columns.str.strip()
best = df.loc[df['metrics/mAP50-95(B)'].idxmax()]
print(f"epochs ที่รัน: {len(df)} · best epoch {int(best['epoch'])}: "
      f"mAP50 {best['metrics/mAP50(B)']:.4f} · mAP50-95 {best['metrics/mAP50-95(B)']:.4f}")
display(Image(filename=str(run / 'results.png')))

In [ ]:
# 7) ตัด runtime ทิ้งเพื่อหยุดกิน CU (ผลอยู่ใน Drive แล้ว)
from google.colab import runtime
runtime.unassign()